In [1]:
import pandas as pd
from pyrosm import OSM
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np


# Геодезическая длина, подходит для всей территории
# Узбекистана без единой UTM-проекции
from pyproj import Geod


In [2]:
osm = OSM("../data/raw/uzbekistan-261102.osm.pbf")

osm

In [3]:
roads = osm.get_network(
    network_type="all",
    extra_attributes=[
        "surface",
        "tracktype",
        "smoothness",
        "access",
        "vehicle",
        "motor_vehicle",
        "motorcar",
        "4wd_only",
    ],
)

print(f"Total segments: {len(roads):,}")
print("Columns:", roads.columns.tolist())

Total segments: 618,617
Columns: ['access', 'bicycle', 'bicycle_road', 'bridge', 'cycleway', 'foot', 'footway', 'highway', 'int_ref', 'junction', 'lanes', 'lit', 'maxspeed', 'motorcar', 'motorroad', 'motor_vehicle', 'name', 'oneway', 'oneway:bicycle', 'overtaking', 'psv', 'ref', 'service', 'segregated', 'sidewalk', 'smoothness', 'surface', 'tracktype', 'tunnel', 'turn', 'width', 'vehicle', 'visible', 'id', 'timestamp', 'version', 'tags', 'osm_type', 'geometry', 'length']


In [4]:
attributes = [
    "tracktype",
    "smoothness",
    "access",
    "vehicle",
    "motor_vehicle",
    "motorcar",
    "4wd_only",
]

summary = []

for col in attributes:
    if col not in roads.columns:
        continue

    counts = roads[col].value_counts(dropna=False)

    summary.append({
        "attribute": col,
        "known_pct": round(roads[col].notna().mean() * 100, 2),
        "unique_values": roads[col].nunique(),
        "top_values": counts.head(5).to_dict(),
    })

display(pd.DataFrame(summary))

,attribute,known_pct,unique_values,top_values
0,tracktype,0.12,5,"{nan: 617897, 'grade2': 232, 'grade5': 195, 'g..."
1,smoothness,0.81,9,"{nan: 613597, 'good': 2106, 'intermediate': 12..."
2,access,0.44,9,"{nan: 615918, 'private': 2026, 'no': 288, 'yes..."
3,vehicle,0.00,4,"{nan: 618601, 'no': 8, 'destination': 4, 'priv..."
4,motor_vehicle,0.12,6,"{nan: 617845, 'no': 326, 'yes': 274, 'private'..."
5,motorcar,0.00,3,"{nan: 618597, 'no': 17, 'yes': 2, 'private': 1}"


In [5]:
conventional_types = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "unclassified", "unclassified_link",
    "residential", "living_street",
    "service",
}

drivable_types = conventional_types | {"track"}

roads["is_conventional"] = (
    roads["highway"].isin(conventional_types)
)

roads["is_potentially_drivable"] = (
    roads["highway"].isin(drivable_types)
)

roads["is_confirmed_asphalt"] = (
    roads["is_potentially_drivable"]
    & roads["surface"].eq("asphalt")
)

In [6]:
restricted_values = {
    "no",
    "private",
    "agricultural",
    "forestry",
}

access_columns = [
    "access",
    "vehicle",
    "motor_vehicle",
    "motorcar",
]

roads["is_access_restricted"] = False

for col in access_columns:
    if col in roads.columns:
        roads["is_access_restricted"] |= (
            roads[col]
            .fillna("")
            .str.lower()
            .isin(restricted_values)
        )

roads["is_publicly_drivable_candidate"] = (
    roads["is_potentially_drivable"]
    & ~roads["is_access_restricted"]
)

In [7]:
classification_summary = pd.DataFrame({
    "category": [
        "Confirmed asphalt",
        "Conventional roads",
        "Potentially drivable",
        "Publicly drivable candidates",
        "Access restricted",
    ],
    "segments": [
        roads["is_confirmed_asphalt"].sum(),
        roads["is_conventional"].sum(),
        roads["is_potentially_drivable"].sum(),
        roads["is_publicly_drivable_candidate"].sum(),
        roads["is_access_restricted"].sum(),
    ],
})

display(classification_summary)

,category,segments
0,Confirmed asphalt,14300
1,Conventional roads,536239
2,Potentially drivable,571899
3,Publicly drivable candidates,569547
4,Access restricted,2754


In [8]:
track_stats = (
    roads.loc[roads["highway"].eq("track")]
    .groupby(["tracktype", "smoothness"], dropna=False)
    .size()
    .reset_index(name="segments")
    .sort_values("segments", ascending=False)
)

display(track_stats.head(30))

,tracktype,smoothness,segments
19,NaN,NaN,34975
3,grade2,NaN,184
13,grade5,NaN,174
0,grade1,NaN,111
6,grade3,NaN,99
9,grade4,NaN,45
15,NaN,horrible,13
14,NaN,bad,12
16,NaN,intermediate,12
5,grade3,very_bad,8


In [9]:
# Preserve all potentially drivable roads
roads_auto = roads.loc[
    roads["is_potentially_drivable"]
].copy()

# Normalize OSM tags
tag_columns = [
    "surface", "tracktype", "smoothness",
    "access", "vehicle", "motor_vehicle",
    "motorcar", "4wd_only"
]

for col in tag_columns:
    if col not in roads_auto.columns:
        roads_auto[col] = pd.NA

    roads_auto[col] = (
        roads_auto[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Explicit restrictions only
restricted = {"no", "private", "agricultural", "forestry"}

access_cols = [
    "access", "vehicle",
    "motor_vehicle", "motorcar"
]

roads_auto["is_access_restricted"] = (
    roads_auto[access_cols]
    .isin(restricted)
    .any(axis=1)
)

# Additional metadata
roads_auto["surface_known"] = roads_auto["surface"].notna()

roads_auto["is_4wd_only"] = (
    roads_auto["4wd_only"].eq("yes").fillna(False)
)

roads_auto["is_rough_track"] = (
    roads_auto["highway"].eq("track")
    & roads_auto["tracktype"].isin(["grade4", "grade5"])
)

roads_auto["is_confirmed_asphalt"] = (
    roads_auto["surface"].eq("asphalt")
)

print(f"Potentially drivable: {len(roads_auto):,}")
print(f"Restricted: {roads_auto['is_access_restricted'].sum():,}")
print(f"Confirmed asphalt: {roads_auto['is_confirmed_asphalt'].sum():,}")

Potentially drivable: 571,899
Restricted: 2,352
Confirmed asphalt: 14,300


In [10]:
print("Total rows:", len(roads_auto))

if "id" in roads_auto.columns:
    print("Duplicate OSM IDs:", roads_auto["id"].duplicated().sum())

print(
    "Duplicate geometries:",
    roads_auto.geometry.to_wkb().duplicated().sum()
)

print(
    "Invalid geometries:",
    (~roads_auto.geometry.is_valid).sum()
)

print(
    "Empty geometries:",
    roads_auto.geometry.is_empty.sum()
)

Total rows: 571899
Duplicate OSM IDs: 0
Duplicate geometries: 1
Invalid geometries: 0
Empty geometries: 0


In [11]:
validation = (
    roads_auto.groupby("highway")
    .agg(
        segments=("highway", "size"),
        restricted=("is_access_restricted", "sum"),
        asphalt=("is_confirmed_asphalt", "sum"),
        rough_track=("is_rough_track", "sum"),
        four_wd=("is_4wd_only", "sum")
    )
)

display(validation.sort_values("segments", ascending=False))

,segments,restricted,asphalt,rough_track,four_wd
highway,,,,,
residential,209216,54,1816,0,0
service,122790,2137,1520,0,0
living_street,121313,32,756,0,0
unclassified,36918,42,314,0,0
track,35660,45,31,227,0
tertiary,17514,31,1725,0,0
secondary,9739,4,2120,0,0
primary,7738,3,2793,0,0
trunk,5758,1,2582,0,0


In [12]:
# Reuse existing osm = OSM(...)

roads_tags = osm.get_network(
    network_type="all",
    extra_attributes=["tunnel", "bridge"]
)

# Ensure matching ID types
roads_auto["id"] = roads_auto["id"].astype(str)
roads_tags["id"] = roads_tags["id"].astype(str)

tags = roads_tags[
    ["id", "tunnel", "bridge"]
].drop_duplicates("id")

roads_auto = roads_auto.drop(
    columns=["tunnel", "bridge"],
    errors="ignore"
).merge(
    tags,
    on="id",
    how="left"
)

roads_auto = gpd.GeoDataFrame(
    roads_auto,
    geometry="geometry",
    crs=roads.crs
)

roads_auto["is_tunnel"] = (
    roads_auto["tunnel"]
    .astype("string")
    .str.lower()
    .eq("yes")
    .fillna(False)
)

print("Tunnel segments:", roads_auto["is_tunnel"].sum())
print(
    "Tunnel segments with confirmed asphalt:",
    (
        roads_auto["is_tunnel"]
        & roads_auto["is_confirmed_asphalt"].astype(str)
          .str.lower().eq("true")
    ).sum()
)

Tunnel segments: 180
Tunnel segments with confirmed asphalt: 23


In [13]:
print(
    roads_auto.loc[
        roads_auto["id"].eq("244248106"),
        ["id", "highway", "tunnel", "is_tunnel"]
    ].to_string(index=False)
)

       id highway tunnel is_tunnel
244248106   trunk    yes      True


In [14]:
from pathlib import Path

output = Path("../data/processed/roads_classified.gpkg")

output.parent.mkdir(parents=True, exist_ok=True)

roads_auto.to_file(
    output,
    layer="roads_classified",
    driver="GPKG",
    engine="pyogrio"
)

print(f"Exported to: {output.resolve()}")

Exported to: C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\processed\roads_classified.gpkg
